# S06 - Halloween and turn-of-the-month seasonality (calendar)

**Hypothesis.** Equity returns are concentrated in November-April ("sell in May", Bouman &
Jacobsen 2002) and around the turn of the month, i.e. the last trading day and the first three of
the next month (Lakonishok & Smidt 1988; McConnell & Xu 2008). Holding equities only in those
windows and T-bills otherwise should earn most of the premium with less time in the market.

| Variant | Hold the S&P 500 on return day d when ... |
|---|---|
| halloween | d falls in November-April |
| tom | d is the last business day of a month or one of the first three of the next |
| halloween_or_tom | November-April, plus the turn-of-month days in May-October |

**Point-in-time.** The rule depends only on the calendar, which is known in advance. The engine
fills at the next close and earns from the following day, so the signal on day t encodes the
desired position for business day t+2 (Mon-Fri calendar; exchange holidays can shift a window by a
day). Daily check, next-close execution, 5 bp per switch (turn-of-month trades ~24 times a year).

**Sources.** Bouman & Jacobsen (2002), *The Halloween indicator*, AER, SSRN 76248; McConnell &
Xu (2008), *Equity returns at the turn of the month*, FAJ, SSRN 917884; Lakonishok & Smidt (1988),
*Are seasonal anomalies real?*, RFS.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S06', 'halloween_tom', 'seasonality/halloween_tom.ipynb'
PROTOCOL, REBALANCE = 'main', 'daily'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 20


## 1. Data and point-in-time inputs

In [2]:
inputs = {"px": px}
print("calendar rule on", len(px), "trading days")

calendar rule on 23300 trading days


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
BD = pd.offsets.BDay()


def desired(days, rule):
    month = days.month
    halloween = np.isin(month, [11, 12, 1, 2, 3, 4])
    last = (days + BD).month != month
    start = days.values.astype("datetime64[M]").astype("datetime64[D]")
    nth = np.busday_count(start, days.values.astype("datetime64[D]")) + 1
    tom = last | (nth <= 3)
    return {"halloween": halloween, "tom": tom, "halloween_or_tom": halloween | tom}[rule]


def build_signal(inputs, rule="halloween"):
    idx = inputs["px"].index
    return pd.Series(desired(idx + 2 * BD, rule).astype(float), index=idx)


VARIANTS = {r: dict(rule=r) for r in ("halloween", "tom", "halloween_or_tom")}
PARAMS_TEXT = {n: f"{n}; calendar (Mon-Fri), signal for t+2 bdays; daily; next_close; 5bp" for n in VARIANTS}

In [4]:
r = px["tr_close"].pct_change(fill_method=None)
by_month = r.groupby(r.index.month).mean() * 21
fig = go.Figure(go.Bar(x=by_month.index, y=by_month, marker_color=["#1f77b4" if m in (11, 12, 1, 2, 3, 4) else "#ff7f0e" for m in by_month.index]))
fig.update_layout(title="Average monthly total return by calendar month, full sample (blue = Nov-Apr)", yaxis_tickformat=".1%", height=340)
fig.show()

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
halloween,30,0,0.0,True
tom,30,0,0.0,True
halloween_or_tom,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
halloween,0.478643,-0.165093,-0.032356,0.489803,2.004065,1935-02-08
tom,0.960348,0.316612,0.449349,0.180202,23.994127,1935-02-08
halloween_or_tom,0.739521,0.095786,0.228523,0.580666,12.024391,1935-02-08


chosen on dev Sharpe only: tom | tom; calendar (Mon-Fri), signal for t+2 bdays; daily; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.960,0.511,0.449,0.644,0.317,0.518,0.442
val,0.335,0.319,0.016,0.572,-0.237,0.309,0.026
holdout,0.251,0.781,-0.530,0.658,-0.407,0.779,-0.528
full,0.669,0.513,0.155,0.630,0.039,0.515,0.154


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.103152,0.064943,0.960348,1.528519,-0.153544,0.671805,-0.098935,1973-03-29,1975-05-05,218.866835,0.180202,1317,23.994127,0.011997
val,0.061782,0.078189,0.335153,0.473728,-0.173667,0.355747,-0.089734,2002-03-04,2004-10-29,3.315589,0.180845,480,24.006573,0.012003
holdout,0.030402,0.071814,0.250615,0.355198,-0.152039,0.199962,-0.04835,2011-04-29,2019-07-02,1.653345,0.181345,403,24.004525,0.012002
full,0.08039,0.069334,0.66851,1.002942,-0.173667,0.462896,-0.098935,2002-03-04,2004-10-29,1199.786847,0.180552,2200,23.993013,0.011997


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.960,0.317,0.449,0.040
val,0.335,-0.237,0.016,0.835
holdout,0.251,-0.407,-0.530,NaN
full,0.669,0.039,0.155,NaN


n_trials=23 | DSR=1.000 | passes=False | holdout confirms=False


dev_beats_sma200         True
dev_beats_bh             True
dev_p_le_0.1             True
val_beats_sma200        False
val_beats_bh             True
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos    False
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.836,0.093,-0.169,24.077
5,0.664,0.080,-0.174,24.077
10,0.490,0.067,-0.179,24.077
25,-0.027,0.029,-0.525,24.077
50,-0.847,-0.031,-0.954,24.077


break-even one-way cost vs SMA200 Sharpe over the same window (0.630): 6.0 bp


,placebo
stat,Sharpe
real,0.663551
placebo_mean,0.167593
placebo_q95,0.32866
quantile,1.0
p_one_sided,0.0
n_placebo,200
time_in_market,0.180472


peek test: Sharpe by shift {-21: 0.526, -5: -0.226, -1: 0.555, 0: 0.664, 1: 0.604, 5: -0.105, 21: 0.556} | gain -0.138


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1930s,1.20,0.26,0.41,0.14,0.03,0.05,0.18
1940s,1.14,0.61,0.57,0.08,0.09,0.06,0.18
1950s,1.85,1.42,1.51,0.12,0.19,0.18,0.18
1960s,0.82,0.40,0.43,0.08,0.08,0.07,0.18
1970s,0.33,0.03,0.37,0.08,0.06,0.09,0.18
1980s,0.78,0.53,0.58,0.14,0.17,0.17,0.18
1990s,0.69,0.92,0.64,0.09,0.18,0.13,0.18
2000s,0.10,-0.05,0.50,0.03,-0.01,0.08,0.18
2010s,0.16,0.90,0.71,0.01,0.14,0.09,0.18
2020s,0.37,0.67,0.58,0.06,0.15,0.10,0.18


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
55,S06-v1/dev,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"halloween; calendar (Mon-Fri), signal for t+2 ...",dev,0.4786,-0.1651,-0.0324,,,,,run,"dev-only sweep, variant halloween"
56,S06-v2/dev,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"tom; calendar (Mon-Fri), signal for t+2 bdays;...",dev,0.9603,0.3166,0.4493,,,,,run,"dev-only sweep, variant tom"
57,S06-v3/dev,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"halloween_or_tom; calendar (Mon-Fri), signal f...",dev,0.7395,0.0958,0.2285,,,,,run,"dev-only sweep, variant halloween_or_tom"
58,S06/dev,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"tom; calendar (Mon-Fri), signal for t+2 bdays;...",dev,0.9603,0.3166,0.4493,0.0405,,23,False,fail,fails frozen metric; chosen tom on dev; protoc...
59,S06/val,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"tom; calendar (Mon-Fri), signal for t+2 bdays;...",val,0.3352,-0.2371,0.0161,0.8354,,23,False,fail,fails frozen metric; chosen tom on dev; protoc...
60,S06/holdout,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"tom; calendar (Mon-Fri), signal for t+2 bdays;...",holdout,0.2506,-0.4069,-0.5301,,,23,False,fail,fails frozen metric; chosen tom on dev; protoc...
61,S06/full,2026-10-20,halloween_tom,seasonality/halloween_tom.ipynb,"tom; calendar (Mon-Fri), signal for t+2 bdays;...",full,0.6685,0.0388,0.1552,,1.0000,23,False,fail,fails frozen metric; chosen tom on dev; protoc...


registry trial count: 23


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (tom) on 2026-10-19: equity weight 0.00


,equity weight at decision
Date,
2026-10-12,0.0
2026-10-13,0.0
2026-10-14,0.0
2026-10-15,0.0
2026-10-16,0.0
2026-10-19,0.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain -0.138.
* dev: Sharpe 0.960 (vs SMA200 +0.317, vs B&H +0.449, p vs SMA200 0.041)
* val: Sharpe 0.335 (vs SMA200 -0.237, vs B&H +0.016, p vs SMA200 0.835)
* holdout: Sharpe 0.251 (vs SMA200 -0.407, vs B&H -0.530)
* Deflated Sharpe 1.000 with n_trials=23.
* Placebo: real 0.664 vs placebo 95th pct 0.329 (p 0.000); break-even cost vs SMA200 full 6.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: val_beats_sma200, val_p_le_0.1); holdout does not confirm.


## 10. Caveats

* The chart above uses the whole sample (descriptive only); the rule itself is fixed by the calendar.
* Turn-of-month trading costs ~24 switches a year; the cost table shows how fast the edge erodes.
* Calendar anomalies were widely published in the 1980s-2000s; post-publication decay is expected.
* Mon-Fri calendar without exchange holidays: windows can be off by one trading day.

## 11. Interpretation

* **Turn-of-the-month was the strongest dev result so far**: Sharpe 0.96 vs SMA200 0.64 (bootstrap p 0.04)
  while invested only 18% of the time, far above every random-timing placebo (p 0.000).
* It then **decays**: val Sharpe 0.34 (below SMA200 0.57, p 0.84) and holdout 0.25 (vs 0.66). The decade
  table shows the edge fading through the 1990s and gone after 2000, the textbook pattern of an
  anomaly that was published (late 1980s) and traded away. **Fails** the frozen metric on val.
* Costs are decisive: ~24 switches a year. Sharpe falls from 0.84 at zero cost to 0.66 at 5 bp and
  below zero at 25 bp; the break-even versus SMA200 is only ~6 bp per switch.
* Halloween alone (Nov-Apr) did not beat B&H in dev. The dev sweep picked `tom` because it was
  best on dev; that choice is exactly what the validation period is there to check, and it failed it.